In [1]:
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams

def extract_smarts(filter_type):
    params = FilterCatalogParams()
    params.AddCatalog(filter_type)
    catalog = FilterCatalog(params)
    
    smarts_dict = {}
    for i in range(catalog.GetNumEntries()):
        entry = catalog.GetEntryWithIdx(i)
        smarts_dict[entry.GetDescription()] = entry.GetSmarts()  # ← corregido
    return smarts_dict

In [ ]:
brenk_raw = extract_smarts(FilterCatalogParams.FilterCatalogs.BRENK)
pains_raw  = extract_smarts(FilterCatalogParams.FilterCatalogs.PAINS)

print(f"Brenk: {len(brenk_raw)} patrones")
print(f"PAINS: {len(pains_raw)} patrones")
```

Debería darte algo como:
```
Brenk: 105 patrones
PAINS: 480 patrones

AttributeError: 'FilterCatalogEntry' object has no attribute 'GetSmarts'

In [3]:
"""
pains_checker.py
----------------
Comprueba si uno o varios SMILES contienen subestructuras PAINS
usando el FilterCatalog de RDKit (implementación SMARTS interna).

Uso:
    python pains_checker.py                     # usa los SMILES de ejemplo
    python pains_checker.py "SMILES1" "SMILES2" # pasa SMILES como argumentos
    python pains_checker.py --file smiles.txt   # lee SMILES de un fichero (uno por línea)

Requisitos:
    pip install rdkit
"""

import sys
from rdkit import Chem
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams


# ── Construcción del catálogo PAINS ──────────────────────────────────────────

def build_pains_catalog():
    params = FilterCatalogParams()
    params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
    return FilterCatalog(params)


# ── Análisis de un único SMILES ───────────────────────────────────────────────

def check_smiles(smiles: str, catalog: FilterCatalog) -> dict:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {"smiles": smiles, "valid": False, "pains": False, "matches": []}

    entries = catalog.GetMatches(mol)
    matches = []
    for entry in entries:
        matches.append({
            "filter_name": entry.GetDescription(),   # nombre del patrón PAINS
            "family":      _get_family(entry.GetDescription()),
        })

    return {
        "smiles":  smiles,
        "valid":   True,
        "pains":   len(matches) > 0,
        "matches": matches,
    }


def _get_family(name: str) -> str:
    """
    Infiere la familia del filtro (A/B/C) a partir del nombre.
    Family A: >149 análogos en la librería original → los más fiables.
    Family B: 15-149 análogos.
    Family C: 1-14 análogos → los menos respaldados estadísticamente.
    RDKit no expone la familia directamente, pero la información está
    en el nombre del patrón: los de Family A son los listados en la
    Tabla 11 del paper de Baell & Holloway 2010.
    """
    # Lista de nombres de Family A (los 16 más poblados)
    family_a = {
        "ene_six_het_a", "mannich_a", "hzone_phenol_b", "anil_di_alk_a",
        "quinone_a", "azo_a", "imine_one_a", "mannich_a", "anil_di_alk_b",
        "anil_di_alk_c", "ene_rhod_a", "hzone_phenol_a", "ene_five_het_a",
        "anil_di_alk_e",
    }
    n = name.lower()
    if n in family_a:
        return "A (alta confianza)"
    # Family B aproximada por prefijos comunes
    elif any(n.startswith(p) for p in ["ene_", "hzone_", "pyrrole_", "thio"]):
        return "B/C (confianza moderada-baja)"
    else:
        return "B/C (confianza moderada-baja)"


# ── Informe de resultados ─────────────────────────────────────────────────────

def print_report(results: list):
    sep = "─" * 60
    for r in results:
        print(sep)
        print(f"SMILES : {r['smiles']}")

        if not r["valid"]:
            print("  ⚠  SMILES inválido (RDKit no pudo parsear la molécula)")
            continue

        if not r["pains"]:
            print("  ✓  Sin alertas PAINS")
        else:
            print(f"  ✗  PAINS detectados ({len(r['matches'])} alerta/s):")
            for m in r["matches"]:
                print(f"      • {m['filter_name']}  [{m['family']}]")
            print()
            print("  Recuerda: una alerta PAINS indica riesgo estadístico,")
            print("  no descarte automático. Consulta el paper de Baell 2010")
            print("  y el 'Seven Year Itch' (2017) para contexto.")
    print(sep)


# ── SMILES de ejemplo ─────────────────────────────────────────────────────────

EXAMPLES = [
    ("Rodanina alquilidénica",      "O=C1SC(=O)/C(=C/c2ccccc2)N1"),
    ("4-Ariliden-azlactona",        "O=C1OC(C)(C)/C(=C/c2ccccc2)N1C(=O)c1ccccc1"),
    ("Azlactona saturada",          "O=C1OC(C)(C)C(Cc2ccccc2)N1C(=O)c1ccccc1"),
    ("Catecol",                     "Oc1ccccc1O"),
    ("Quinona",                     "O=C1C=CC(=O)C=C1"),
    ("Imidazolona con farmacóforo", "O=C1N(CC)C(CS(=O)(C)=O)=N/C1=C\c2ccccc2"),
    ("Tiazolona con farmacóforo",   "O=C1SC(CS(=O)(C)=O)=N/C1=C\c2ccccc2"),
    ("Amida (control negativo)",    "CC(=O)Nc1ccccc1"),
]


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    catalog = build_pains_catalog()

    # Modo fichero
    if len(sys.argv) == 3 and sys.argv[1] == "--file":
        with open(sys.argv[2]) as f:
            smiles_list = [line.strip() for line in f if line.strip()]
        results = [check_smiles(s, catalog) for s in smiles_list]
        print_report(results)
        return

    # Modo argumentos en línea de comandos
    if len(sys.argv) > 1:
        smiles_list = sys.argv[1:]
        results = [check_smiles(s, catalog) for s in smiles_list]
        print_report(results)
        return

    # Modo ejemplos por defecto
    print("\nEjemplos de referencia (azlactonas + controles)\n")
    for label, smi in EXAMPLES:
        r = check_smiles(smi, catalog)
        r["label"] = label
        print(f"\n[{label}]")
        r_wrapped = [r]
        print_report(r_wrapped)


if __name__ == "__main__":
    main()


────────────────────────────────────────────────────────────
SMILES : --f=/Users/dariomlorente/Library/Jupyter/runtime/kernel-v36321562c8e026804d72c5477aa9892f6ea3bb8db.json
  ⚠  SMILES inválido (RDKit no pudo parsear la molécula)
────────────────────────────────────────────────────────────


[20:26:14] SMILES Parse Error: syntax error while parsing: --f=/Users/dariomlorente/Library/Jupyter/runtime/kernel-v36321562c8e026804d72c5477aa9892f6ea3bb8db.json
[20:26:14] SMILES Parse Error: Failed parsing SMILES '--f=/Users/dariomlorente/Library/Jupyter/runtime/kernel-v36321562c8e026804d72c5477aa9892f6ea3bb8db.json' for input: '--f=/Users/dariomlorente/Library/Jupyter/runtime/kernel-v36321562c8e026804d72c5477aa9892f6ea3bb8db.json'


In [4]:
"""
pains_checker.py
----------------
Comprueba si uno o varios SMILES contienen subestructuras PAINS
usando el FilterCatalog de RDKit (implementación SMARTS interna).

Uso:
    python pains_checker.py                     # usa los SMILES de ejemplo
    python pains_checker.py "SMILES1" "SMILES2" # pasa SMILES como argumentos
    python pains_checker.py --file smiles.txt   # lee SMILES de un fichero (uno por línea)

Requisitos:
    pip install rdkit
"""

import sys
from rdkit import Chem
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams


# ── Construcción del catálogo PAINS ──────────────────────────────────────────

def build_pains_catalog():
    params = FilterCatalogParams()
    params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
    return FilterCatalog(params)


# ── Análisis de un único SMILES ───────────────────────────────────────────────

def check_smiles(smiles: str, catalog: FilterCatalog) -> dict:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {"smiles": smiles, "valid": False, "pains": False, "matches": []}

    entries = catalog.GetMatches(mol)
    matches = []
    for entry in entries:
        matches.append({
            "filter_name": entry.GetDescription(),   # nombre del patrón PAINS
            "family":      _get_family(entry.GetDescription()),
        })

    return {
        "smiles":  smiles,
        "valid":   True,
        "pains":   len(matches) > 0,
        "matches": matches,
    }


def _get_family(name: str) -> str:
    """
    Infiere la familia del filtro (A/B/C) a partir del nombre.
    Family A: >149 análogos en la librería original → los más fiables.
    Family B: 15-149 análogos.
    Family C: 1-14 análogos → los menos respaldados estadísticamente.
    RDKit no expone la familia directamente, pero la información está
    en el nombre del patrón: los de Family A son los listados en la
    Tabla 11 del paper de Baell & Holloway 2010.
    """
    # Lista de nombres de Family A (los 16 más poblados)
    family_a = {
        "ene_six_het_a", "mannich_a", "hzone_phenol_b", "anil_di_alk_a",
        "quinone_a", "azo_a", "imine_one_a", "mannich_a", "anil_di_alk_b",
        "anil_di_alk_c", "ene_rhod_a", "hzone_phenol_a", "ene_five_het_a",
        "anil_di_alk_e",
    }
    n = name.lower()
    if n in family_a:
        return "A (alta confianza)"
    # Family B aproximada por prefijos comunes
    elif any(n.startswith(p) for p in ["ene_", "hzone_", "pyrrole_", "thio"]):
        return "B/C (confianza moderada-baja)"
    else:
        return "B/C (confianza moderada-baja)"


# ── Informe de resultados ─────────────────────────────────────────────────────

def print_report(results: list):
    sep = "─" * 60
    for r in results:
        print(sep)
        print(f"SMILES : {r['smiles']}")

        if not r["valid"]:
            print("  ⚠  SMILES inválido (RDKit no pudo parsear la molécula)")
            continue

        if not r["pains"]:
            print("  ✓  Sin alertas PAINS")
        else:
            print(f"  ✗  PAINS detectados ({len(r['matches'])} alerta/s):")
            for m in r["matches"]:
                print(f"      • {m['filter_name']}  [{m['family']}]")
            print()
            print("  Recuerda: una alerta PAINS indica riesgo estadístico,")
            print("  no descarte automático. Consulta el paper de Baell 2010")
            print("  y el 'Seven Year Itch' (2017) para contexto.")
    print(sep)


# ── SMILES de ejemplo ─────────────────────────────────────────────────────────

EXAMPLES = [
    ("Rodanina alquilidénica",      "O=C1SC(=O)/C(=C/c2ccccc2)N1"),
    ("4-Ariliden-azlactona",        "O=C1OC(C)(C)/C(=C/c2ccccc2)N1C(=O)c1ccccc1"),
    ("Azlactona saturada",          "O=C1OC(C)(C)C(Cc2ccccc2)N1C(=O)c1ccccc1"),
    ("Catecol",                     "Oc1ccccc1O"),
    ("Quinona",                     "O=C1C=CC(=O)C=C1"),
    ("Imidazolona con farmacóforo", "O=C1N(CC)C(CS(=O)(C)=O)=N/C1=C\c2ccccc2"),
    ("Tiazolona con farmacóforo",   "O=C1SC(CS(=O)(C)=O)=N/C1=C\c2ccccc2"),
    ("Amida (control negativo)",    "CC(=O)Nc1ccccc1"),
]


# ── Main ──────────────────────────────────────────────────────────────────────

def _is_jupyter_arg(arg: str) -> bool:
    """Detecta argumentos del kernel de Jupyter/IPython en sys.argv."""
    return (
        arg.startswith("--f=")
        or arg.startswith("--ip=")
        or arg.startswith("--stdin=")
        or arg.startswith("--control=")
        or arg.startswith("--hb=")
        or arg.startswith("--Session.")
        or "kernel-" in arg
        or arg.endswith(".json")
    )


def _get_user_args() -> list:
    """Devuelve solo los argumentos que son SMILES reales, ignorando Jupyter."""
    return [a for a in sys.argv[1:] if not _is_jupyter_arg(a)]


def main():
    catalog = build_pains_catalog()
    user_args = _get_user_args()

    # Modo fichero
    if len(user_args) == 2 and user_args[0] == "--file":
        with open(user_args[1]) as f:
            smiles_list = [line.strip() for line in f if line.strip()]
        results = [check_smiles(s, catalog) for s in smiles_list]
        print_report(results)
        return

    # Modo argumentos en línea de comandos
    if user_args:
        results = [check_smiles(s, catalog) for s in user_args]
        print_report(results)
        return

    # Modo ejemplos por defecto
    print("\nEjemplos de referencia (azlactonas + controles)\n")
    for label, smi in EXAMPLES:
        r = check_smiles(smi, catalog)
        r["label"] = label
        print(f"\n[{label}]")
        print_report([r])


if __name__ == "__main__":
    main()


Ejemplos de referencia (azlactonas + controles)


[Rodanina alquilidénica]
────────────────────────────────────────────────────────────
SMILES : O=C1SC(=O)/C(=C/c2ccccc2)N1
  ✓  Sin alertas PAINS
────────────────────────────────────────────────────────────

[4-Ariliden-azlactona]
────────────────────────────────────────────────────────────
SMILES : O=C1OC(C)(C)/C(=C/c2ccccc2)N1C(=O)c1ccccc1
  ✓  Sin alertas PAINS
────────────────────────────────────────────────────────────

[Azlactona saturada]
────────────────────────────────────────────────────────────
SMILES : O=C1OC(C)(C)C(Cc2ccccc2)N1C(=O)c1ccccc1
  ✓  Sin alertas PAINS
────────────────────────────────────────────────────────────

[Catecol]
────────────────────────────────────────────────────────────
SMILES : Oc1ccccc1O
  ✗  PAINS detectados (1 alerta/s):
      • catechol_A(92)  [B/C (confianza moderada-baja)]

  Recuerda: una alerta PAINS indica riesgo estadístico,
  no descarte automático. Consulta el paper de Baell 2010
  y el